# 02. TorchAO Modern API and Affine Quantized Tensor (AQT)

- **Category:** 14_quantisation / 02_pt2e_torchao_and_bitpacking
- **Lineage & Citations:** PyTorch Architecture Optimization Team (2024, *torchao: Native Architecture Optimization and Low-Precision Quantization in PyTorch*); Dettmers et al. (2022, *LLM.int8()*); Frantar et al. (2022, *GPTQ*).
- **Purpose:** Formulate the modern `torchao` quantization API, composable tensor subclassing via `__torch_dispatch__`, the `AffineQuantizedTensor` (AQT) block-wise layout, Weight-Only quantization (`Int8WeightOnlyConfig`, `Int4WeightOnlyConfig`), Dynamic W8A8 integer GEMM, and execution regime crossovers.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [01. PT2E Export Quantization Workflow](01_pt2e_export_quantization_workflow.ipynb) $\to$ `🔗 Downstream Syntheses:` [03. Sub Byte Bitpacking T-MAC LUT and GGUF K Quants](03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use Weight-Only W4A16 at large batch sizes ($B \ge 128$) where GEMMs are compute-bound (dequantization overhead makes W4A16 *slower* than BF16; use dynamic W8A8 instead!); do NOT subclass `nn.Module` when tensor subclasses (`AffineQuantizedTensor`) preserve graph exportability.

## Mathematical Foundations & Affine Quantized Tensor Mechanics

1. **Affine Quantized Tensor (AQT) Block-Wise Layout**:
   $$s_g = \frac{\max(|W_g|)}{q_{\max}}, \quad q_g = \operatorname{clamp}\left(\left\lfloor \frac{W_g}{s_g} \right\rceil, q_{\min}, q_{\max}\right), \quad \hat{W}_g = q_g \cdot s_g$$
2. **Weight-Only Quantization (W4A16 / W8A16)**:
   $$Y = X \cdot \hat{W} + b \implies \text{HBM reading volume reduced by } 2\text{--}4\times \text{ for small batch decoding } (B \le 8)$$
3. **Dynamic W8A8 Mixed-Scale Integer GEMM**:
   $$s_X[b] = \frac{\max_j |X_{b, j}|}{127}, \quad s_W[c] = \frac{\max_i |W_{i, c}|}{127}, \quad Y = (X_q \cdot W_q) \odot (s_X \cdot s_W) + b$$
4. **Regime Crossover Theorem**:
   $$\text{Arithmetic Intensity } I = \frac{2 B D_{\text{in}} D_{\text{out}}}{2 D_{\text{in}} D_{\text{out}} \cdot \text{BytesPerParam} + 2 B D_{\text{in}} \cdot 2} \implies \begin{cases} B \le 8 & \text{Memory-bound (W4/W8 wins)} \\ B \ge 64 & \text{Compute-bound (W8A8/FP8 wins)} \end{cases}$$

In [ ]:
import sys, time, math
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/14_quantisation/02_pt2e_torchao_and_bitpacking/02_torchao_modern_api_and_affine_quantized_tensor.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def linear_anchor(x, weight, bias):
    return x @ weight + bias

def affine_quantize_blockwise(weight, group_size=16, qmin=-128.0, qmax=127.0):
    orig_shape = weight.shape
    flat = weight.flatten()
    n_groups = len(flat) // group_size
    scales, q_ints, dequants = [], [], []
    for g in range(n_groups):
        chunk = flat[g * group_size : (g + 1) * group_size]
        amax = float(torch.max(torch.abs(chunk)))
        scale = amax / qmax if amax > 0.0 else 1.0
        q = torch.clamp(torch.round(chunk / scale), qmin, qmax)
        scales.append(scale)
        q_ints.append(q)
        dequants.append(q * scale)
    return torch.cat(dequants).reshape(orig_shape), torch.tensor(scales), torch.cat(q_ints).reshape(orig_shape)

def dynamic_w8a8_linear(x, weight, bias):
    s_x = torch.max(torch.abs(x), dim=-1, keepdim=True).values / 127.0
    s_x = torch.where(s_x == 0.0, torch.ones_like(s_x), s_x)
    x_q = torch.clamp(torch.round(x / s_x), -128.0, 127.0)
    s_w = torch.max(torch.abs(weight), dim=0, keepdim=True).values / 127.0
    s_w = torch.where(s_w == 0.0, torch.ones_like(s_w), s_w)
    w_q = torch.clamp(torch.round(weight / s_w), -128.0, 127.0)
    return (x_q @ w_q) * (s_x * s_w) + bias, s_x, s_w

def int4_weight_only_linear(x, weight, bias, group_size=16):
    w_deq, scales, _ = affine_quantize_blockwise(weight, group_size=group_size, qmin=-8.0, qmax=7.0)
    return x @ w_deq + bias, w_deq, scales

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert torch.max(torch.abs(y_anchor - weights["test_output"])) < 1e-6

# 2. Block-wise INT8 AQT Parity
w_deq, w_scales, w_int = affine_quantize_blockwise(weights["w_linear"], group_size=16, qmin=-128.0, qmax=127.0)
assert torch.max(torch.abs(w_deq - weights["w_block_dequant"])) < 1e-5
assert torch.max(torch.abs(w_scales - weights["w_block_scales"])) < 1e-5
assert torch.max(torch.abs(w_int - weights["w_block_int"])) < 1e-5
out_w = weights["x_act"] @ w_deq + weights["b_linear"]
assert torch.max(torch.abs(out_w - weights["out_w_only"])) < 1e-5

# 3. Dynamic W8A8 Linear Parity
y_w8a8, s_x, s_w = dynamic_w8a8_linear(weights["x_act"], weights["w_linear"], weights["b_linear"])
assert torch.max(torch.abs(y_w8a8 - weights["w8a8_out"])) < 1e-5
assert torch.max(torch.abs(s_x - weights["w8a8_s_x"])) < 1e-5
assert torch.max(torch.abs(s_w - weights["w8a8_s_w"])) < 1e-5

# 4. INT4 Weight-Only Linear Parity
out_w4, w4_deq, w4_scales = int4_weight_only_linear(weights["x_act"], weights["w_linear"], weights["b_linear"], group_size=16)
assert torch.max(torch.abs(out_w4 - weights["out_w4_only"])) < 1e-5
assert torch.max(torch.abs(w4_deq - weights["w4_block_dequant"])) < 1e-5
assert torch.max(torch.abs(w4_scales - weights["w4_block_scales"])) < 1e-5
print("All TorchAO AQT, W8A8, and W4A16 parity assertions verified!")

In [ ]:
# Microbenchmark: Decoding Regime (B=1) vs Prefill Regime (B=64)
x_dec = weights["x_act"][:1]
t0 = time.perf_counter()
for _ in range(200): _ = int4_weight_only_linear(x_dec, weights["w_linear"], weights["b_linear"], group_size=16)
t_dec_w4 = (time.perf_counter() - t0) / 200 * 1e6

x_pref = weights["x_act"].repeat(16, 1)
t0 = time.perf_counter()
for _ in range(200): _ = dynamic_w8a8_linear(x_pref, weights["w_linear"], weights["b_linear"])
t_pref_w8a8 = (time.perf_counter() - t0) / 200 * 1e6

print(f"Decoding B=1  (W4A16 latency):  {t_dec_w4:.2f} us (HBM bandwidth bound)")
print(f"Prefill  B=64 (W8A8 latency):   {t_pref_w8a8:.2f} us (Compute bound)")

In [ ]:
# Failure Mode: Compute-Bound Large Batch (B=256) Misuse of Weight-Only W4 vs Dynamic W8A8
torch.manual_seed(42)
x_huge = torch.randn(256, 16)

# FP32 Baseline ground truth
fp32_ref = x_huge @ weights["w_linear"] + weights["b_linear"]

# 1. Weight-Only W4A16
out_w4, _, _ = int4_weight_only_linear(x_huge, weights["w_linear"], weights["b_linear"], group_size=16)
mse_w4 = float(torch.mean((fp32_ref - out_w4) ** 2))
sqnr_w4 = 10.0 * math.log10(float(torch.mean(fp32_ref ** 2)) / mse_w4)

# 2. Dynamic W8A8
out_w8a8, _, _ = dynamic_w8a8_linear(x_huge, weights["w_linear"], weights["b_linear"])
mse_w8a8 = float(torch.mean((fp32_ref - out_w8a8) ** 2))
sqnr_w8a8 = 10.0 * math.log10(float(torch.mean(fp32_ref ** 2)) / mse_w8a8)

print(f"W4A16 SQNR at B=256: {sqnr_w4:.2f} dB (4-bit truncation)")
print(f"W8A8  SQNR at B=256: {sqnr_w8a8:.2f} dB (Higher precision 8-bit)")
advantage = sqnr_w8a8 - sqnr_w4
print(f"W8A8 Precision Advantage: +{advantage:.2f} dB over W4A16 in large batch regimes!")
assert advantage > 15.0, "W8A8 must beat W4A16 by >15 dB SQNR in large-batch prefill!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Tensor Subclass Decoupling**: `torchao`'s `AffineQuantizedTensor` packages scales and zero-points directly into tensor metadata, bypassing brittle module surgery and preserving standard PyTorch dispatch.
2. **Serving Regime Alignment**: Deploy Weight-Only W4A16 for memory-bandwidth-bound single-stream generation ($B \le 8$), and dynamic W8A8 / FP8 for compute-bound high-concurrency prefill ($B \ge 64$).
3. **Block-Wise Adaptivity**: Group-wise scaling ($G=16\text{--}64$) confines activation and weight outliers to localized blocks, preventing dynamic range collapse across entire channels.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_torchao_modern_api_and_affine_quantized_tensor.md](../../../markdown/14_quantisation/02_pt2e_torchao_and_bitpacking/02_torchao_modern_api_and_affine_quantized_tensor.md)
- **PyTorch 2.x Module:** [02_torchao_modern_api_and_affine_quantized_tensor.py](../../../pytorch2/14_quantisation/02_pt2e_torchao_and_bitpacking/02_torchao_modern_api_and_affine_quantized_tensor.py)
- **Functional JAX Module:** [02_torchao_modern_api_and_affine_quantized_tensor.py](../../../jax/14_quantisation/02_pt2e_torchao_and_bitpacking/02_torchao_modern_api_and_affine_quantized_tensor.py)
- **Native Rust Candle Track:** [02_torchao_modern_api_and_affine_quantized_tensor.rs](../../../candle/14_quantisation/02_pt2e_torchao_and_bitpacking/02_torchao_modern_api_and_affine_quantized_tensor.rs)
- **Hardware Kernel (CPU / WGSL):** [02_torchao_modern_api_and_affine_quantized_tensor.rs](../../../kernels/14_quantisation/02_pt2e_torchao_and_bitpacking/02_torchao_modern_api_and_affine_quantized_tensor.rs)
- **Golden SafeTensors:** [02_torchao_modern_api_and_affine_quantized_tensor.safetensors](../../../weights/14_quantisation/02_pt2e_torchao_and_bitpacking/02_torchao_modern_api_and_affine_quantized_tensor.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. PT2E Export Quantization Workflow](01_pt2e_export_quantization_workflow.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 03. Sub Byte Bitpacking T-MAC LUT and GGUF K Quants](03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.ipynb) &rarr;